# JagX-AI — Kaggle GPU pretraining

This notebook downloads a real open training corpus, prepares deterministic train/validation data, trains a JagX tokenizer, verifies that packed training batches exist, and starts native JagX causal-LM pretraining on the Kaggle GPU.

Default: 50,000 OASST1 records and 1,000 optimizer steps. Increase `JAGX_ROWS`/`JAGX_STEPS` after the first successful run.

In [ ]:
import os, subprocess, sys, pathlib

# Kaggle Settings -> Accelerator -> GPU must be enabled.
print('Python:', sys.version)

REPO = pathlib.Path('/kaggle/working/JagX-AI')
if not REPO.exists():
    subprocess.check_call(['git', 'clone', 'https://github.com/Tajudeen001-security/JagX-AI.git', str(REPO)])
else:
    subprocess.check_call(['git', '-C', str(REPO), 'fetch', 'origin', 'main'])
    subprocess.check_call(['git', '-C', str(REPO), 'reset', '--hard', 'origin/main'])
os.chdir(REPO)
print('Repo:', pathlib.Path.cwd())
print('Commit:', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())

In [ ]:
# Install JagX plus the Hugging Face dataset client.
!pip install -q -e . datasets huggingface_hub

import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('GPU is not enabled. Open Kaggle Settings, select a GPU accelerator, and restart the session.')

In [ ]:
# End-to-end training. All large data/checkpoints stay in Kaggle's working storage, not GitHub.
# The launcher now checks for a real packed batch and uses a re-iterable batch stream,
# so multi-step training can safely restart at the end of each corpus pass.
os.environ.setdefault('JAGX_ROWS', '50000')
os.environ.setdefault('JAGX_STEPS', '1000')
os.environ.setdefault('JAGX_SEQ_LEN', '512')
os.environ.setdefault('JAGX_BATCH_SIZE', '4')
os.environ.setdefault('JAGX_GRAD_ACCUM', '8')

!python scripts/kaggle_train.py --skip-pip

In [ ]:
# Inspect the resulting checkpoints.
from pathlib import Path
ckpt = Path('kaggle_checkpoints')
print('Checkpoint directory:', ckpt.resolve())
for p in sorted(ckpt.glob('*.pt')):
    print(p.name, f'{p.stat().st_size / (1024**2):.1f} MB')